# 🧹 Level 7: Data Cleaning

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_7_Data_Cleaning)**

In real life, data is messy.
- Missing values (NaN)
- Outliers (Crazy values)

If we don't clean it, our model learns garbage!

### 1. Load Data & Create Messy Samples
We will intentionally break our clean sample dataset to simulate real-world problems.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)

np.random.seed(42)
# Add missing values
mask = np.random.rand(len(df)) < 0.05  # 5% missing
df.loc[mask, 'area_m2'] = np.nan

# Add huge outliers
mask_outlier = np.random.rand(len(df)) < 0.01
df.loc[mask_outlier, 'price_10k_krw'] = df['price_10k_krw'] * 10

print("Messy Data Created!")
df.info()

### 2. Handle Missing Values
Check for NaNs.

In [ ]:
print(df.isnull().sum())

Strategy: Fill missing **Area** with the **Median** value.

In [ ]:
median_area = df['area_m2'].median()
df['area_m2'].fillna(median_area, inplace=True)
print("Missing values filled.")

### 3. Handle Outliers (IQR Method)
Outliers can ruin Linear Regression. Let's find them.

In [ ]:
sns.boxplot(x=df['price_10k_krw'])
plt.title("Price Box Plot (With Outliers)")
plt.show()

In [ ]:
Q1 = df['price_10k_krw'].quantile(0.25)
Q3 = df['price_10k_krw'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Keep prices between {lower_bound:,.0f} and {upper_bound:,.0f}")

# Filter
df_clean = df[(df['price_10k_krw'] >= lower_bound) & (df['price_10k_krw'] <= upper_bound)]
print(f"Original rows: {len(df)}, Clean rows: {len(df_clean)}")

### 4. Train Model on Clean Data
Now we train on the clean dataset.

In [ ]:
X = df_clean[['area_m2']].values
y = df_clean['price_10k_krw'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f"RMSE with Clean Data: {rmse:,.0f}")